# Cycle Analysis — 260121 CZA-flueCO2 Experiment

Validate the cycle detection and integration pipeline on the 260121 dataset.

In [2]:
import sys
from pathlib import Path

# Add backend to path so we can import effi
sys.path.insert(0, str(Path.cwd().parent / "backend"))

import pandas as pd
import plotly.graph_objects as go
from effi import (
    load_experiment,
    detect_cycles,
    analyze_experiment,
)
from effi.integration import NATIVE_SPECIES

## 1. Load Data

In [3]:
data_dir = Path.cwd().parent / "260121_CZA-flueCO2"

reactor_files = sorted(data_dir.glob("ExportData*.txt"))
reactor_files = [str(f) for f in reactor_files]
print(f"Reactor files: {len(reactor_files)}")
for f in reactor_files:
    print(f"  {Path(f).name}")

ir_file = str(data_dir / "260121_Data_All.csv")
oxygen_file = str(data_dir / "260121_oxygen.csv")

df = load_experiment(
    reactor_files,
    ir_file,
    oxygen_file,
    offset=pd.Timedelta("5h"),
)
print(f"\nMerged DataFrame: {df.shape[0]} rows x {df.shape[1]} columns")
print(f"Time range: {df['Timestamp'].min()} to {df['Timestamp'].max()}")

Reactor files: 4
  ExportData_20260110155819_20260122155903.txt
  ExportData_20260112110215_20260122155936.txt
  ExportData_20260113170209_20260121143403.txt
  ExportData_20260116201025_20260121143239.txt

Merged DataFrame: 36838 rows x 126 columns
Time range: 2026-01-10 20:58:41 to 2026-01-21 14:28:07


## 2. Detect Cycles

In [4]:
cycles = detect_cycles(df)
print(f"Detected {len(cycles)} cycles\n")

# Print summary table
cycle_summary = []
for c in cycles:
    cycle_summary.append({
        "cycle": c.cycle_id,
        "hp_start": c.high_p.start,
        "hp_end": c.high_p.end,
        "lp_start": c.low_p.start,
        "lp_end": c.low_p.end,
        "hp_rows": c.high_p.end_idx - c.high_p.start_idx + 1,
        "lp_rows": c.low_p.end_idx - c.low_p.start_idx + 1,
    })
pd.DataFrame(cycle_summary)

Detected 35 cycles



,cycle,hp_start,hp_end,lp_start,lp_end,hp_rows,lp_rows
0,1,2026-01-10 21:14:03,2026-01-12 15:27:36,2026-01-12 15:27:36,2026-01-12 21:11:27,6102,829
1,2,2026-01-12 17:17:39,2026-01-12 20:08:20,2026-01-12 20:08:20,2026-01-12 21:11:27,412,153
2,3,2026-01-12 23:17:42,2026-01-13 02:08:22,2026-01-13 02:08:22,2026-01-13 03:11:29,412,153
3,4,2026-01-13 05:17:44,2026-01-13 08:08:25,2026-01-13 08:08:25,2026-01-13 09:11:32,412,153
4,5,2026-01-13 11:17:46,2026-01-13 14:08:27,2026-01-13 14:08:27,2026-01-13 15:11:34,412,153
5,6,2026-01-13 17:17:49,2026-01-13 20:08:05,2026-01-13 20:08:05,2026-01-13 21:11:37,411,154
6,7,2026-01-13 23:17:27,2026-01-14 02:08:07,2026-01-14 02:08:07,2026-01-14 03:11:14,412,153
7,8,2026-01-14 05:17:53,2026-01-14 08:08:09,2026-01-14 08:08:09,2026-01-14 09:11:16,411,153
8,9,2026-01-14 11:17:31,2026-01-14 14:08:12,2026-01-14 14:08:12,2026-01-14 15:11:19,412,153
9,10,2026-01-14 17:17:33,2026-01-14 20:08:14,2026-01-14 20:08:14,2026-01-14 21:11:21,412,153


## 3. Integrate Species

In [5]:
results = analyze_experiment(df, cycles)
print(f"Results: {results.shape[0]} rows\n")

# Check for negative areas or NaN
n_negative = (results[["high_p_area", "low_p_area"]] < 0).sum().sum()
n_nan = results[["high_p_area", "low_p_area"]].isna().sum().sum()
print(f"Negative areas: {n_negative}")
print(f"NaN areas: {n_nan}\n")

# Show results for key product species
key_species = ["Methanol", "Ethanol", "Dimethyl Ether", "Carbon Dioxide", "Water"]
results[results["species"].isin(key_species)]

Results: 525 rows

Negative areas: 0
NaN areas: 0



,cycle_id,species,unit,high_p_area,low_p_area
4,1,Carbon Dioxide,%·s,9435.528926,8690.944810
5,1,Dimethyl Ether,%·s,0.244760,0.014189
6,1,Water,%·s,12451.974897,5926.606691
7,1,Methanol,%·s,72.432908,72.566424
9,1,Ethanol,ppm·s,320958.215108,52154.644462
...,...,...,...,...,...
514,35,Carbon Dioxide,%·s,7484.834313,19.601551
515,35,Dimethyl Ether,%·s,0.044471,0.013323
516,35,Water,%·s,3920.625738,501.376118
517,35,Methanol,%·s,0.000000,0.000000


## 4. Visualize Cycles

Plot a single cycle with fill-between shading for high-P and low-P windows.

In [6]:
def plot_cycle(df, cycle, species=None):
    """Plot one cycle with window shading."""
    if species is None:
        species = ["Methanol (%)", "Dimethyl Ether (%)", "Carbon Dioxide (%)"]

    # Pad view by 2 minutes on each side
    pad = pd.Timedelta("2min")
    t_start = cycle.high_p.start - pad
    t_end = cycle.low_p.end + pad
    mask = (df["Timestamp"] >= t_start) & (df["Timestamp"] <= t_end)
    view = df.loc[mask]

    fig = go.Figure()

    # High-P shading
    fig.add_vrect(
        x0=cycle.high_p.start, x1=cycle.high_p.end,
        fillcolor="rgba(0,100,255,0.1)", line_width=0,
        annotation_text="High P", annotation_position="top left",
    )
    # Low-P shading
    fig.add_vrect(
        x0=cycle.low_p.start, x1=cycle.low_p.end,
        fillcolor="rgba(255,100,0,0.1)", line_width=0,
        annotation_text="Low P", annotation_position="top left",
    )

    for col in species:
        if col in view.columns:
            fig.add_trace(go.Scatter(
                x=view["Timestamp"], y=view[col],
                mode="lines", name=col,
            ))

    # Add reactor conditions on secondary y-axis
    for col, dash in [("Reactor P RSP", "dash"), ("Reactor T RSP", "dot")]:
        if col in view.columns:
            fig.add_trace(go.Scatter(
                x=view["Timestamp"], y=view[col],
                mode="lines", name=col,
                line=dict(dash=dash),
                yaxis="y2",
            ))

    fig.update_layout(
        title=f"Cycle {cycle.cycle_id}",
        xaxis_title="Time",
        yaxis=dict(title="Concentration (%)"),
        yaxis2=dict(title="RSP (°C / bar)", overlaying="y", side="right"),
        hovermode="x unified",
        height=500,
    )
    return fig

In [7]:
# Verify 3 cycles: early, middle, late
for idx in [0, len(cycles) // 2, len(cycles) - 1]:
    c = cycles[idx]
    print(f"\nCycle {c.cycle_id}: HP {c.high_p.start} → {c.high_p.end}, "
          f"LP {c.low_p.start} → {c.low_p.end}")
    fig = plot_cycle(df, c)
    fig.show()


Cycle 1: HP 2026-01-10 21:14:03 → 2026-01-12 15:27:36, LP 2026-01-12 15:27:36 → 2026-01-12 21:11:27



Cycle 18: HP 2026-01-17 02:26:06 → 2026-01-17 05:16:21, LP 2026-01-17 05:16:21 → 2026-01-17 06:19:29



Cycle 35: HP 2026-01-21 08:25:59 → 2026-01-21 11:16:15, LP 2026-01-21 11:16:15 → 2026-01-21 12:19:48


## 5. Sanity Checks

In [8]:
# Check STATUS columns are unaffected
status_cols = [c for c in df.columns if "STATUS" in c]
for col in status_cols:
    unique_vals = df[col].dropna().unique()
    print(f"{col}: unique values = {sorted(unique_vals)}")

print()

# Check that integration results are positive and reasonable
pivot = results.pivot_table(
    index="cycle_id",
    columns="species",
    values="high_p_area",
)
print("High-P area summary (all cycles):")
pivot[["Methanol", "Ethanol", "Dimethyl Ether"]].describe()

HPLC pump STATUS: unique values = [np.float64(0.0)]
Reactor bypass STATUS: unique values = [np.float64(0.0)]
Cond. bypass STATUS: unique values = [np.float64(0.0), np.float64(1.0)]
CO2 bypass STATUS: unique values = [np.float64(1.0)]
HB BLOW STATUS: unique values = [np.float64(1.0)]

High-P area summary (all cycles):


species,Methanol,Ethanol,Dimethyl Ether
count,35.000000,35.000000,35.000000
mean,10.742460,35471.431005,0.046388
std,16.881865,50495.984942,0.043198
min,0.000000,9165.024227,0.000000
25%,0.047975,21463.066939,0.027022
50%,3.616755,27212.912262,0.031437
75%,14.502289,30931.912110,0.060479
max,72.432908,320958.215108,0.244760
